# 04 — Preprocesamiento y tokenización

> **Objetivo.** Convertir pares `(src, tgt)` en tensores listos para entrenar T5. Cubre: prefijo de tarea, tokenización SentencePiece, padding dinámico vs estático, etiquetas (labels) y `DataCollatorForSeq2Seq`.

## Conceptos

### Prefijo de tarea
T5 distingue tareas por prefijo de texto. Para GEC:
```
entrada modelo  =  "grammar: " + src
salida modelo   =  tgt
```
El prefijo es **arbitrario pero consistente**. La PoC sigue la convención `"grammar: "`.

### Tokenización SentencePiece
T5 usa **SentencePiece** (BPE-like) con un vocabulario de 32 128 tokens. Cada palabra se descompone en subpalabras (subwords) — manejando OOVs y morfología.

### Padding dinámico vs estático
| Estrategia | Cuándo se aplica | Costo |
|---|---|---|
| **`padding="max_length"`** | en `tokenizer(...)` | siempre 128 tokens, desperdicia compute en oraciones cortas |
| **`padding=False` + DataCollator** | en runtime, por batch | paddea al máximo del batch — eficiente |

La guía original usa estática; la PoC usa **dinámica** (decisión documentada en `docs/decisiones_tecnicas.md` ADR-05).

### Labels
Para seq2seq cross-entropy, los `labels` son los `input_ids` del target. Tokens de padding deben ser **enmascarados con `-100`** para que la loss los ignore. `DataCollatorForSeq2Seq` hace esto automáticamente.

In [ ]:
# === CONFIG ===
MODEL_NAME       = "google-t5/t5-small"
MAX_INPUT_LEN    = 128
MAX_TARGET_LEN   = 128
PREFIX           = "grammar: "
# ==============

from pathlib import Path
from datasets import load_from_disk
from transformers import AutoTokenizer

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
SPLITS_PATH = ROOT / "artifacts" / "coedit_gec_splits"

## 1. Cargar splits filtrados (output del notebook 03)

In [ ]:
splits = load_from_disk(str(SPLITS_PATH))
print(splits)
print("\nEjemplo:")
print(splits["train"][0])

## 2. Cargar el tokenizer

Usamos `AutoTokenizer.from_pretrained` que selecciona automáticamente la versión *fast* (Rust-backed) del tokenizer T5.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
print("Tokenizer:", type(tokenizer).__name__)
print("Vocab size:", tokenizer.vocab_size)
print("Pad token id:", tokenizer.pad_token_id)
print("EOS token id:", tokenizer.eos_token_id)

## 3. Inspección — cómo tokeniza T5

Vemos cómo SentencePiece descompone una oración corta. El símbolo `▁` representa el espacio que precede al token.

In [ ]:
demo = "grammar: I has a apple."
toks = tokenizer.tokenize(demo)
ids  = tokenizer.encode(demo)
print("Texto :", demo)
print("Tokens:", toks)
print("IDs   :", ids)
print("Decode:", tokenizer.decode(ids, skip_special_tokens=False))

## 4. Validar `max_length` con la distribución real

Queremos confirmar que `MAX_INPUT_LEN=128` cubre el p99 del dataset filtrado tras tokenizar.

In [ ]:
import numpy as np

sample = splits["train"].select(range(min(2000, len(splits["train"]))))
src_lens = [len(tokenizer.encode(PREFIX + ex["src"])) for ex in sample]
tgt_lens = [len(tokenizer.encode(ex["tgt"])) for ex in sample]

print(f"SRC tokens — p50: {np.percentile(src_lens, 50):.0f}  p95: {np.percentile(src_lens, 95):.0f}  p99: {np.percentile(src_lens, 99):.0f}  max: {max(src_lens)}")
print(f"TGT tokens — p50: {np.percentile(tgt_lens, 50):.0f}  p95: {np.percentile(tgt_lens, 95):.0f}  p99: {np.percentile(tgt_lens, 99):.0f}  max: {max(tgt_lens)}")

## 5. Función de preprocesamiento

La función:
1. Antepone `"grammar: "` a cada `src`.
2. Tokeniza inputs y targets sin padding (lo hará el collator).
3. Trunca a `max_length`.

Usamos `text_target=...` (API moderna desde `transformers>=4.21`), no `tokenizer.as_target_tokenizer()` que está deprecado.

In [ ]:
def preprocess(batch):
    inputs = [PREFIX + s for s in batch["src"]]
    model_inputs = tokenizer(
        inputs,
        max_length=MAX_INPUT_LEN,
        truncation=True,
        padding=False,           # padding dinámico vía collator
    )
    labels = tokenizer(
        text_target=batch["tgt"],
        max_length=MAX_TARGET_LEN,
        truncation=True,
        padding=False,
    )
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

tokenized = splits.map(
    preprocess,
    batched=True,
    remove_columns=splits["train"].column_names,
    desc="Tokenizando",
)
print(tokenized)

In [ ]:
# Verificar primer ejemplo
ex = tokenized["train"][0]
print("input_ids[:20]:", ex["input_ids"][:20])
print("labels[:20]   :", ex["labels"][:20])
print()
print("INPUT decoded :", tokenizer.decode(ex["input_ids"], skip_special_tokens=True))
print("LABEL decoded :", tokenizer.decode(ex["labels"], skip_special_tokens=True))

## 6. DataCollator — padding dinámico + máscara de labels

`DataCollatorForSeq2Seq` se encarga de:

1. Paddear `input_ids` y `attention_mask` al máximo del batch.
2. Paddear `labels` al máximo del batch usando **`-100`** (no `pad_token_id`) → la cross-entropy ignora esos tokens automáticamente.
3. Ofrecer multiplicador `pad_to_multiple_of=8` (útil en GPUs con tensor cores; en MPS no aporta tanto pero es seguro).

In [ ]:
from transformers import DataCollatorForSeq2Seq, T5ForConditionalGeneration

# Necesitamos una referencia al modelo (para que el collator detecte el tipo)
model_ref = T5ForConditionalGeneration.from_pretrained(MODEL_NAME)

collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model_ref,
    padding=True,
    label_pad_token_id=-100,
    pad_to_multiple_of=8,
)

# Demo: paddear un batch de 4 ejemplos
batch = collator([tokenized["train"][i] for i in range(4)])
for k, v in batch.items():
    print(f"{k:18s} shape={tuple(v.shape)} dtype={v.dtype}")

In [ ]:
# Verificar que pad tokens en labels = -100
import torch
labels = batch["labels"]
n_pad   = (labels == -100).sum().item()
n_total = labels.numel()
print(f"Tokens enmascarados (-100) en labels: {n_pad}/{n_total} ({n_pad/n_total:.1%})")

## 7. Persistir tokenized dataset

El notebook 05 carga directamente desde aquí — fine-tuning no debe re-tokenizar.

In [ ]:
OUT = ROOT / "artifacts" / "tokenized_dataset"
tokenized.save_to_disk(str(OUT))
print("Guardado en:", OUT)

## Resumen

- Prefijo: `"grammar: "`
- Truncado a 128/128 tokens (cubre p99).
- Padding dinámico vía `DataCollatorForSeq2Seq`.
- Labels paddeados con `-100` para enmascarar la loss.

**Siguiente → `05_finetuning.ipynb`**
